# Competição · 2/4 · modelos bayesianos em R-INLA (BB oficial e GHR/BSC)

**Antes de rodar (uma vez só):** em `MyDrive/dengue_V4/` (a pasta do V4, que já tem `dados/` e `referencias_imdc3/`) suba `comp_lib.py`, `comp_modelos.py` e `comp_inla.R`, e **substitua o `v4_lib.py`** pela versão nova de `data_imdc_2026/V4/`.
Todos os resultados vão para `MyDrive/dengue_V4/resultados_competicao/`. Se a sessão cair, rode de novo: o que já foi feito é pulado.

| Notebook | Modelos | Ambiente |
|---|---|---|
| `Comp1_CPU` | climatologia, canal_endemico, AR_longo, SURGE, ARIMA_Fourier, ETS_MSTL, Prophet, LightGBM_direto, Renovacao_S, Tamanho_x_forma | CPU |
| `Comp2_INLA` | BB_INLA, GHR_INLA | CPU com bastante RAM |
| `Comp3_GPU` | XGBoost_quantis, DeepAR, NHITS, TFT, TSMixer, Chronos, AGCRN | GPU L4 |
| `Comp4_Avaliacao` | rankings, gráficos, diagnóstico, ensembles | qualquer (CPU) |

Os notebooks 1, 2 e 3 podem rodar **ao mesmo tempo**, cada um numa sessão. O 4 pode rodar a qualquer momento: ele avalia o que já existir.

**Ambiente:** CPU com bastante RAM (em *Alterar tipo de ambiente*, escolha **High-RAM**).

**Como está otimizado:**
- ajuste só com as linhas observadas (6 temporadas de treino por região de saúde);
- modo `compact` do INLA (o mais rápido) com estratégia `gaussian` e `eb`;
- amostras só dos efeitos, com o preditor montado no R.

**BB:** semana e ano por macrorregião do país (5), como no original. **GHR:** BYM2 no grafo das 435 regiões + semana e ano por UF + clima.

**Primeiro rode só 1 temporada** (`TESTE_RAPIDO = True`) para medir o tempo. Depois rode o resto. Na primeira sessão, o INLA é instalado no disco local do Colab e uma cópia compactada fica no Drive (`R_libs_inla.tar.gz`); nas próximas sessões ele é restaurado em ~1 min. A célula de instalação termina com um **teste do motor** (`MOTOR_OK`): se ele falhar, não adianta rodar o resto.

In [ ]:
NOTEBOOK = 'inla'
PACOTES = ''

# @title Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/comp', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'comp_inla.R', 'dados', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_competicao')
os.makedirs(os.path.join(PASTA_RES, 'graficos'), exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM
for m in (V, L, CM):
    importlib.reload(m)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ctx = L.Contexto(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'))
TODAS = list(range(2016, 2026))          # 2016–2021: calibração dos pesos do ensemble · 2022–2025: S1–S4
RECENTES = list(range(2020, 2026))       # para os modelos mais caros
OFICIAIS = V.TEMPORADAS_OFICIAIS
M = 500                                  # trajetórias por região
REFAZER = False

In [ ]:
# @title Instalar o R-INLA (só na primeira vez; cópia compactada fica no Drive)
# O Drive não deixa executar programas (o motor inla.mkl.run dá "Permission denied").
# Por isso o INLA roda do disco local do Colab (/content/R_libs); o Drive só guarda um .tar.gz,
# que preserva as permissões de execução.
R_LIBS = '/content/R_libs'; os.makedirs(R_LIBS, exist_ok=True)
os.environ['R_LIBS_USER'] = R_LIBS
TAR_DRIVE = os.path.join(PASTA_DRIVE, 'R_libs_inla.tar.gz')
ANTIGO_DRIVE = os.path.join(PASTA_DRIVE, 'R_libs')          # instalação antiga, direto no Drive


def tem_inla():
    r = subprocess.run(['Rscript', '-e', 'cat(requireNamespace("INLA", quietly=TRUE))'], capture_output=True, text=True)
    return 'TRUE' in r.stdout


def libera_execucao():
    b = os.path.join(R_LIBS, 'INLA', 'bin')
    if os.path.isdir(b):
        subprocess.run(['chmod', '-R', 'a+rx', b])


if shutil.which('Rscript') is None:
    print('Instalando o R (~2 min)...'); subprocess.run('apt-get -qq update && apt-get -qq install -y r-base-core > /dev/null', shell=True)
print(subprocess.run(['Rscript', '--version'], capture_output=True, text=True).stderr.split('\n')[0])
if not tem_inla() and os.path.exists(TAR_DRIVE):
    print('Restaurando o INLA do Drive (~1 min)...'); subprocess.run(['tar', '-xzf', TAR_DRIVE, '-C', '/content'])
if not tem_inla() and os.path.isdir(os.path.join(ANTIGO_DRIVE, 'INLA')):
    print('Copiando a instalação antiga do Drive para o disco local (~2 min)...')
    subprocess.run(f'cp -r "{ANTIGO_DRIVE}"/. "{R_LIBS}"/', shell=True)
libera_execucao()
if not tem_inla():
    print('Instalando o INLA (10–20 min, só na primeira vez)...')
    script = r'''
    lib <- Sys.getenv("R_LIBS_USER"); options(timeout = 3600, Ncpus = 4)
    options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
    install.packages(c("Matrix", "sp", "fmesher", "foreach", "withr", "MatrixModels"), lib = lib)
    ok <- function() requireNamespace("INLA", quietly = TRUE)
    try(install.packages("INLA", lib = lib, repos = c(getOption("repos"), INLA = "https://inla.r-inla-download.org/R/stable"), dependencies = FALSE))
    if (!ok()) try(install.packages("https://inla.r-inla-download.org/R/stable/src/contrib/INLA_25.10.19.tar.gz", lib = lib, repos = NULL, type = "source"))
    if (!ok()) try(install.packages("https://inla.r-inla-download.org/R/stable/src/contrib/INLA_25.06.07.tar.gz", lib = lib, repos = NULL, type = "source"))
    cat("INLA:", ok(), "\n")'''
    r = subprocess.run(['Rscript', '-e', script], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    libera_execucao()
print('INLA disponível:', tem_inla())

# Pacotes que o INLA usa em tempo de execução ('sn' é exigido pelo inla.posterior.sample)
EXTRAS = ['sn', 'mnormt', 'numDeriv', 'quantreg', 'SparseM', 'MatrixModels']
falta = subprocess.run(['Rscript', '-e', 'p <- commandArgs(TRUE); cat(p[!sapply(p, requireNamespace, quietly=TRUE)])', *EXTRAS],
                       capture_output=True, text=True).stdout.split()
if falta:
    print('Instalando pacotes auxiliares:', falta)
    subprocess.run(['Rscript', '-e', 'install.packages(commandArgs(TRUE), lib=Sys.getenv("R_LIBS_USER"), '
                    'repos="https://packagemanager.posit.co/cran/__linux__/jammy/latest")', *falta], capture_output=True)
    if os.path.exists(TAR_DRIVE):
        os.remove(TAR_DRIVE)                                  # regrava a cópia do Drive já com os auxiliares
if tem_inla() and not os.path.exists(TAR_DRIVE):
    print('Guardando cópia compactada no Drive para as próximas sessões...')
    subprocess.run(['tar', '-czf', '/content/R_libs_inla.tar.gz', '-C', '/content', 'R_libs'])
    shutil.copy('/content/R_libs_inla.tar.gz', TAR_DRIVE)

# Teste de fumaça: o motor do INLA precisa rodar de verdade (não basta o pacote carregar)
teste = r'''
suppressMessages(library(INLA)); set.seed(1)
d <- data.frame(y = rpois(50, 5), x = rnorm(50))
r <- inla(y ~ x, family = "poisson", data = d, inla.mode = "compact", control.compute = list(config = TRUE))
a <- inla.posterior.sample(5, r)
cat("MOTOR_OK", round(r$summary.fixed[1, "mean"], 2), "| amostras:", length(a), "| versão", as.character(packageVersion("INLA")), "\n")'''
r = subprocess.run(['Rscript', '-e', teste], capture_output=True, text=True)
print(r.stdout[-500:] if 'MOTOR_OK' in r.stdout else ('❌ O motor do INLA não rodou:\n' + r.stdout[-1500:] + r.stderr[-2500:]))

In [ ]:
TESTE_RAPIDO = True            # True: roda só a temporada 2022 do BB para medir o tempo; depois mude para False
kw = dict(anos=6, M=300, threads=os.cpu_count(), r_libs=R_LIBS, pasta_script=PASTA_LOCAL)
MODELOS = {'BB_INLA': CM.ModeloINLA('bb', **kw), 'GHR_INLA': CM.ModeloINLA('ghr', **kw)}
PLANO = {'BB_INLA': [2022]} if TESTE_RAPIDO else {'BB_INLA': RECENTES, 'GHR_INLA': RECENTES}

In [ ]:
# @title Rodar (retoma de onde parou; um modelo com erro não interrompe os outros)
erros = []
for nome, temps in PLANO.items():
    for s in temps:
        try:
            dt = L.rodar(ctx, MODELOS[nome], s, PASTA_RES, M=M, refazer=REFAZER)
            if dt is not None:
                print(f'✅ {nome:18s} {s}: {dt / 60:5.1f} min', flush=True)
        except Exception as e:
            erros.append((nome, s, repr(e)))
            print(f'❌ {nome:18s} {s}: {repr(e)[:400]}', flush=True)
            traceback.print_exc(limit=2)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'temporada', 'erro']).to_csv(os.path.join(PASTA_RES, f'erros_{NOTEBOOK}.csv'), index=False)
t = L.tempos(PASTA_RES)
t[t.modelo.isin(list(PLANO))].groupby('modelo').segundos.agg(['count', 'mean', 'sum']).div([1, 60, 60]).round(1) \
    .rename(columns={'count': 'temporadas', 'mean': 'min/temporada', 'sum': 'min total'})

In [ ]:
# @title Tempo do INLA por temporada (modo usado e segundos de ajuste)
for f in sorted(glob.glob(os.path.join(PASTA_RES, 'log', '*INLA*.json'))):
    d = json.load(open(f)); print(d['modelo'], d['season'], round(d['segundos']), 's ·', d.get('tempo_R', '').replace('\n', ' '))